# Structured Outputs for Scientific Workflows

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

LLMs are great at reading. They are bad at *handing structured data to your code*.

If you've ever asked GPT or Claude for "JSON" and gotten back

```
Sure! Here is the JSON you asked for:
```json
{...}
```
Let me know if you need anything else!
```

…you know the problem. This notebook shows the three patterns that actually work in production:

1. **Tool-based forced JSON** — the most reliable trick: pretend the schema is a tool the model must call.
2. **Pydantic validation + retry loop** — catch malformed output and ask the model to fix it.
3. **Schema-driven extraction over noisy scientific text** — turn a free-form synthesis paragraph into a clean record your downstream pipeline can consume.

## Why this matters for science

Almost every agentic scientific workflow eventually hits a step like:
- *"Extract the calcination temperature from this 200-page methods PDF."*
- *"Parse the user's free-text request into reconstruction parameters."*
- *"Read the instrument's text log and emit a structured event."*

If that step is flaky, the whole agent is flaky. Structured outputs are the seam between *language* and *code*.

---
## Setup

In [ ]:
# !pip install anthropic pydantic

In [ ]:
import os, getpass, json
from typing import Optional, Literal

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

import anthropic
from pydantic import BaseModel, Field, ValidationError

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"  # latest Sonnet as of 2026

---
## The naive approach: ask for JSON in the prompt

Let's start with the temptation everyone gives in to: just *ask* the model for JSON.

In [ ]:
SAMPLE_TEXT = '''
We synthesized BaTiO3 nanoparticles by a sol-gel route. Barium acetate (Ba(CH3COO)2,
99.99%) and titanium isopropoxide were dissolved in 2-methoxyethanol and stirred at
60 C for 4 hours. The resulting gel was dried at 120 C overnight and calcined at 750 C
for 6 hours under flowing oxygen. XRD confirmed phase-pure tetragonal BaTiO3 with an
average crystallite size of 38 nm (Scherrer analysis on the (101) reflection).
'''

resp = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": (
            "Extract the synthesis details as JSON with fields: "
            "material, route, precursors, calcination_temp_C, calcination_time_h, atmosphere.\n\n"
            f"Text:\n{SAMPLE_TEXT}"
        )
    }],
)
print(resp.content[0].text)

You'll often get *something that looks like* JSON, sometimes wrapped in
prose, sometimes inside a Markdown code fence, sometimes with stray comments.
Now imagine running this 10,000 times in a pipeline. **You will spend more
time writing regex than science.**

---
## Pattern 1: Forced JSON via tool use

The trick: define a "tool" whose only purpose is to *receive* the structured data.
The model has no choice but to fill in the schema you specify. This is the same
mechanism agents use to call functions — we're just abusing it for extraction.

In [ ]:
SYNTHESIS_TOOL = {
    "name": "record_synthesis",
    "description": "Record a single materials-synthesis procedure extracted from text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "material":               {"type": "string", "description": "Target material formula"},
            "route":                  {"type": "string",
                                       "enum": ["sol-gel", "solid-state", "hydrothermal",
                                                "co-precipitation", "CVD", "other"]},
            "precursors":             {"type": "array", "items": {"type": "string"}},
            "calcination_temp_C":     {"type": "number"},
            "calcination_time_h":     {"type": "number"},
            "atmosphere":             {"type": "string"},
            "crystallite_size_nm":    {"type": ["number", "null"]},
        },
        "required": ["material", "route", "precursors",
                     "calcination_temp_C", "calcination_time_h", "atmosphere"],
    },
}

resp = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[SYNTHESIS_TOOL],
    tool_choice={"type": "tool", "name": "record_synthesis"},   # FORCE the tool call
    messages=[{
        "role": "user",
        "content": f"Extract the synthesis from this passage:\n\n{SAMPLE_TEXT}"
    }],
)

# The structured output lives in the tool_use block, already parsed as a dict
tool_use = next(b for b in resp.content if b.type == "tool_use")
record = tool_use.input
print(json.dumps(record, indent=2))

**What's different:**
- `tool_choice={"type": "tool", "name": "..."}` *forces* the model to call this specific tool.
- `tool_use.input` is already a Python dict — Anthropic parsed it for us.
- The schema is enforced server-side. Required fields can't be missing. Enums can't drift.

---
## Pattern 2: Pydantic validation + retry loop

Even with forced tool use, things can still go wrong:
- A free-text field contains nonsense (`calcination_time_h = "overnight"`)
- The model invents a value not actually in the source text
- Custom business rules (e.g. "calcination temp must be > 200 C") aren't expressible in JSON Schema

Define a Pydantic model — same shape, plus richer validation — and retry on failure.

In [ ]:
class Synthesis(BaseModel):
    material: str
    route: Literal["sol-gel", "solid-state", "hydrothermal",
                   "co-precipitation", "CVD", "other"]
    precursors: list[str] = Field(min_length=1)
    calcination_temp_C: float = Field(gt=200, lt=2000)
    calcination_time_h: float = Field(gt=0, lt=240)
    atmosphere: str
    crystallite_size_nm: Optional[float] = Field(default=None, gt=0, lt=10_000)

def extract_with_retry(text: str, max_attempts: int = 3) -> Synthesis:
    messages = [{
        "role": "user",
        "content": f"Extract the synthesis from this passage:\n\n{text}"
    }]
    for attempt in range(max_attempts):
        resp = client.messages.create(
            model=MODEL, max_tokens=1024,
            tools=[SYNTHESIS_TOOL],
            tool_choice={"type": "tool", "name": "record_synthesis"},
            messages=messages,
        )
        tool_use = next(b for b in resp.content if b.type == "tool_use")
        try:
            return Synthesis(**tool_use.input)
        except ValidationError as e:
            print(f"[attempt {attempt+1}] validation failed: {e.error_count()} errors")
            # Feed the error back so the model can self-correct
            messages.append({"role": "assistant", "content": resp.content})
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": f"Validation failed:\n{e}\nPlease re-extract.",
                    "is_error": True,
                }],
            })
    raise RuntimeError("Could not produce a valid Synthesis after retries")

record = extract_with_retry(SAMPLE_TEXT)
print(record.model_dump_json(indent=2))

Notice the retry pattern: we feed the **validation error message** back to the model
as a tool result. The model sees its own attempt and the specific complaint, and tries
again. This is the same loop pattern used by full agents — just with one tool.

---
## Pattern 3: Batch extraction over messy real-world text

Let's process several abstracts and ask the model to *also* tell us when it's
unsure. Adding a `confidence` and `evidence` field is one of the highest-ROI
moves in scientific extraction — you get a built-in flag for human review.

In [ ]:
ABSTRACTS = [
    # Clean, complete
    "We report sol-gel synthesis of SrTiO3 from Sr(NO3)2 and Ti(O-iPr)4 in ethanol; "
    "the gel was calcined at 800 C for 4 h in air, yielding 25 nm cubic perovskite "
    "particles confirmed by XRD.",
    # Missing temperature, hand-wave-y
    "BiFeO3 thin films were prepared by a wet chemical route from bismuth and iron "
    "nitrates dissolved in 2-methoxyethanol. After spin-coating, films were annealed "
    "at moderate temperatures under flowing oxygen.",
    # Wrong domain entirely (should fail / low confidence)
    "We trained a graph neural network on the Materials Project dataset using a "
    "batch size of 64 and the Adam optimizer with a learning rate of 1e-4.",
]

class SynthesisWithEvidence(Synthesis):
    confidence: Literal["high", "medium", "low"]
    evidence_quote: str = Field(description="Verbatim quote supporting the extraction")
    extraction_notes: Optional[str] = None

EVIDENCE_TOOL = {
    "name": "record_synthesis_with_evidence",
    "description": "Record an extracted synthesis along with confidence and supporting quote.",
    "input_schema": SynthesisWithEvidence.model_json_schema(),
}

# Pydantic emits 'allOf'/'$ref' for inheritance — Anthropic wants a flat schema.
# For tutorial simplicity we override with a hand-flattened schema:
EVIDENCE_TOOL["input_schema"] = {
    "type": "object",
    "properties": {
        **SYNTHESIS_TOOL["input_schema"]["properties"],
        "confidence":       {"type": "string", "enum": ["high", "medium", "low"]},
        "evidence_quote":   {"type": "string"},
        "extraction_notes": {"type": ["string", "null"]},
    },
    "required": SYNTHESIS_TOOL["input_schema"]["required"] + ["confidence", "evidence_quote"],
}

results = []
for i, abstract in enumerate(ABSTRACTS):
    resp = client.messages.create(
        model=MODEL, max_tokens=1024,
        tools=[EVIDENCE_TOOL],
        tool_choice={"type": "tool", "name": "record_synthesis_with_evidence"},
        system=("You are a careful scientific information extractor. "
                "If a value is not stated in the text, set crystallite_size_nm=null "
                "and use 'low' confidence. Never invent numbers."),
        messages=[{"role": "user", "content": f"Extract:\n\n{abstract}"}],
    )
    tu = next(b for b in resp.content if b.type == "tool_use")
    results.append(tu.input)
    print(f"--- Abstract {i+1} ({tu.input.get('confidence','?').upper()}) ---")
    print(json.dumps(tu.input, indent=2))
    print()

The third abstract is *not* a synthesis at all. With a good system prompt,
the model should flag it with `low` confidence and a note. In practice, this is
the difference between a pipeline that silently corrupts your dataset and one
that surfaces uncertainty for review.

---
## Takeaways

| Pattern | When to use |
|---|---|
| Forced tool call | The default for any "give me JSON" task. Don't ask in prose. |
| Pydantic + retry | When validation rules go beyond JSON Schema, or when downstream code is brittle to bad input. |
| Confidence + evidence | Whenever the source text is messy and you need a "human review" flag. |

**Anti-patterns to avoid:**
- Parsing JSON out of free-form prose with regex.
- Trusting `response_format` / `strict: true` flags as a substitute for downstream validation.
- Forcing a schema so rigid the model has to lie to fill it.

## Next up

The next notebook (RAG over scientific literature) shows how to *ground*
extractions like these in real source documents instead of trusting the model's
internal knowledge.